# 05 · Content-based filtering

The collaborative model (notebook 04, frozen as tag `cf-baseline`) is strong on well-known movies but can't score movies with no positive ratings, and rarely recommends rare ones. This notebook builds a model that recommends from what a movie is (genres, year, later tags and TMDB metadata) instead of who rated it, then combines it with the collaborative model.

We use the same 17,241 movies as notebook 04. Movies dropped by the 5-rating filter are not in this catalogue.

In [22]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "scripts"))   # notebooks/ -> project root -> scripts

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from recutils import load_split, to_sets

NAME, K, REL, SEED = "sample_10pct", 10, 4.0, 42

S = load_split(NAME)
train, val, test = S["train"], S["val"], S["test"]
n_users = int(S["user_map"].u.max()) + 1
n_items = int(S["item_map"].i.max()) + 1
print(f"{n_users:,} users | {n_items:,} movies in the modelled catalogue")

movies = pd.read_parquet(Path("../data/interim") / NAME / "movies.parquet")
meta   = S["item_map"].merge(movies, on="movieId", how="left").set_index("i").sort_index()
assert (meta.index.values == np.arange(n_items)).all() and meta.title.notna().all()

baseline = pd.read_csv("../reports/cf_baseline_test.csv", index_col=0)
baseline.round(4)

20,093 users | 17,241 movies in the modelled catalogue


,Recall@10,NDCG@10,HitRate@10,Coverage
popularity (all ratings),0.0531,0.0393,0.1562,0.0147
popularity (ratings >= 4),0.0524,0.0396,0.1564,0.0148
"popularity (all ratings), users with pos.",0.0531,0.0393,0.1563,0.0147
[nb 03] explicit MF,0.0020,0.0012,0.0072,0.2628
"implicit ALS (liked, k=64, a=5, lam=80), all users",0.1193,0.0878,0.3144,0.0904
"implicit ALS (same), users with pos.",0.1193,0.0878,0.3145,0.0904


## 1 · How much of the evaluation do rare movies make up?
We sort movies by how many users rated them 4 or higher in the training data ("positives"). A movie with few positives gives a collaborative model very little to learn from. A held-out rating of 4 or higher counts as "relevant", meaning a recommendation we'd want to get right.

The table shows, for each group, how many movies it contains and how many relevant validation and test interactions fall on it. This tells us whether rarely rated movies matter enough to the metrics for a content model to show any effect, and whether there are enough users among them to judge it fairly.

In [23]:
n_pos = np.bincount(train.i[train.rating >= REL], minlength=n_items)

bins   = [-1, 0, 4, 19, 99, 499, np.inf]
labels = ["0", "1-4", "5-19", "20-99", "100-499", "500+"]
bucket = pd.Series(pd.cut(n_pos, bins=bins, labels=labels), name="bucket")

def share_table(rel_df):
    b = pd.Series(bucket.values[rel_df.i.values])
    n_int = b.value_counts().reindex(labels)
    n_usr = pd.DataFrame({"u": rel_df.u.values, "b": b.values}).groupby("b", observed=False)["u"].nunique().reindex(labels)
    return n_int, n_usr

val_rel, test_rel = val[val.rating >= REL], test[test.rating >= REL]
v_int, v_usr = share_table(val_rel)
t_int, t_usr = share_table(test_rel)

summary = pd.DataFrame({
    "movies": bucket.value_counts().reindex(labels),
    "% of catalogue": (100 * bucket.value_counts().reindex(labels) / n_items).round(1),
    "relevant val interactions": v_int,
    "% of relevant val": (100 * v_int / v_int.sum()).round(2),
    "users (val)": v_usr,
    "relevant test interactions": t_int,
    "% of relevant test": (100 * t_int / t_int.sum()).round(2),
    "users (test)": t_usr,
})
summary

,movies,% of catalogue,relevant val interactions,% of relevant val,users (val),relevant test interactions,% of relevant test,users (test)
0,1095,6.4,80,0.11,77,121,0.16,116
1-4,5666,32.9,784,1.04,625,985,1.28,796
5-19,4836,28.0,2355,3.13,1761,2924,3.80,2220
20-99,3299,19.1,8460,11.23,5345,9646,12.54,6041
100-499,1685,9.8,21356,28.34,10663,22204,28.86,11236
500+,660,3.8,42318,56.16,14988,41049,53.36,15018


## 1b · How much metadata do rare movies have?
A content model can only use what's recorded about a movie. For each group, the table shows the share of movies with genres, the share with a release year, and the average number of genres per movie. If rare movies have little metadata, a content model can't help them. If movies have only about 2 genres each, many will look alike to it.

In [24]:
meta["year"]      = meta.title.str.extract(r"\((\d{4})\)\s*$")[0].astype("float")
meta["has_genre"] = meta.genres.ne("(no genres listed)")
meta["has_year"]  = meta.year.notna()
meta["n_genres"]  = meta.genres.where(meta.has_genre, "").str.count(r"\|").add(1).where(meta.has_genre, 0)
meta["bucket"]    = bucket.values

cov = meta.groupby("bucket", observed=False).agg(
    movies=("title", "size"),
    pct_with_genre=("has_genre", "mean"),
    pct_with_year=("has_year", "mean"),
    mean_n_genres=("n_genres", "mean"),
)
cov[["pct_with_genre", "pct_with_year"]] = (100 * cov[["pct_with_genre", "pct_with_year"]]).round(1)
cov["mean_n_genres"] = cov["mean_n_genres"].round(2)
cov.reindex(labels)

,movies,pct_with_genre,pct_with_year,mean_n_genres
bucket,,,,
0,1095,98.7,99.8,2.07
1-4,5666,98.9,99.7,1.98
5-19,4836,99.3,99.7,2.02
20-99,3299,99.8,99.7,2.22
100-499,1685,99.8,99.8,2.51
500+,660,100.0,100.0,2.92


**Observations**
- A small group of well-known movies carries most of the evaluation. The 660 movies with 500+ positives (3.8% of the catalogue) hold 56% of the relevant validation interactions. A positive is a training rating of 4 or higher; a relevant interaction is a held-out rating of 4 or higher.
- Movies with fewer than 20 positives are 67% of the catalogue but only 4.3% of the relevant validation interactions (5.2% on test). Movies with fewer than 5 positives are 39% of the catalogue and about 1.2% of the interactions (1.4% on test). The collaborative model has almost nothing to learn from for these.
- Almost every movie has metadata: at least 98.7% have genres and 99.7% have a year, in every group. But rare movies carry less of it, around 2 genres each against 2.9 for the most-rated group.

**What that might mean**
- Even a perfect model for rare movies could only touch about 4 to 5% of the relevant interactions. Overall NDCG will barely move when we add content features, so a change there wouldn't tell us much. The content model has to be judged on rare movies separately.
- Genres and year reach nearly every rare movie, but with about two genres each many movies will look alike to the model. We'll check how alike in the next step.

**Decisions (made now, before seeing any content-model results)**
- *Cold slice*: movies with fewer than 5 positives. *Tail slice*: fewer than 20. Tail is the main slice because it has enough users to measure; cold is reported next to it as the hardest case. Movies with zero positives (77 validation users) are too few to judge alone.
- Two ways to evaluate on a slice: (A) the normal top-10 over all movies, counting how many rare relevant movies show up, and (B) ranking only the slice's movies against each other, to see which model orders rare movies best.
- For test, slice membership is recomputed with train + validation, because that is the data the final model sees.
- Metrics are averaged over users with at least one relevant movie in the slice, with a 95% confidence interval from resampling users, since the slices are small.

## 1c · Fix the rare-movie slices
This cell turns the decisions above into code: flags marking which movies are in the cold and tail slices (for validation, using training data; for test, using train + validation), and a table of how big each slice is. "Random Recall@10" is what a random ranking would score if it had to pick from only that slice's movies. It's the floor for the in-slice evaluation.

In [25]:
def positives(df):
    return np.bincount(df.i[df.rating >= REL], minlength=n_items)

n_pos_val  = positives(train)                         # what the model knows when we tune on validation
n_pos_test = positives(pd.concat([train, val]))       # what the final model knows when we test

SLICES = {"cold": 5, "tail": 20}                      # fewer than this many positives
in_slice = {("val", s): n_pos_val < t for s, t in SLICES.items()}
in_slice.update({("test", s): n_pos_test < t for s, t in SLICES.items()})

def slice_table(rel_df, split):
    rows = {}
    for s, t in SLICES.items():
        movies_in = in_slice[(split, s)]
        hit = movies_in[rel_df.i.values]
        rows[f"{s} (<{t} positives)"] = {
            "movies": int(movies_in.sum()),
            "relevant interactions": int(hit.sum()),
            "% of all relevant": round(100 * hit.mean(), 2),
            "users with >=1": int(np.unique(rel_df.u.values[hit]).size),
            "random Recall@10": round(10 / movies_in.sum(), 4),
        }
    return pd.DataFrame(rows).T.astype({"movies": int, "relevant interactions": int, "users with >=1": int})

print("validation (movie counts from train)")
display(slice_table(val_rel, "val"))
print("test (movie counts from train + validation)")
display(slice_table(test_rel, "test"))

validation (movie counts from train)


,movies,relevant interactions,% of all relevant,users with >=1,random Recall@10
cold (<5 positives),6761,864,1.15,676,0.0015
tail (<20 positives),11597,3219,4.27,2144,0.0009


test (movie counts from train + validation)


,movies,relevant interactions,% of all relevant,users with >=1,random Recall@10
cold (<5 positives),6572,1033,1.34,814,0.0015
tail (<20 positives),11456,3802,4.94,2587,0.0009


## 2 · Turn movies into feature vectors
Genres become a multi-hot vector (one column per genre, 1 if the movie has it). Release decade becomes a one-hot vector. The next cells build both, weight the columns so rare features count more, and scale every row to length 1.

In [26]:
genres = meta.genres.str.get_dummies("|").drop(columns="(no genres listed)", errors="ignore").astype("int8")

decade  = (meta.year // 10 * 10).clip(lower=1910)           # anything before 1920 goes into one bucket
decades = pd.get_dummies(decade.map(lambda d: f"{int(d)}s" if pd.notna(d) else None)).astype("int8")

print("genre matrix:", genres.shape, "|", int(genres.sum(axis=1).eq(0).sum()), "movies with no genre")
print("decade matrix:", decades.shape, "|", int(decades.sum(axis=1).eq(0).sum()), "movies with no year")
genres.sum().sort_values(ascending=False)

genre matrix: (17241, 19) | 125 movies with no genre
decade matrix: (17241, 12) | 47 movies with no year


Drama          8082
Comedy         5819
Thriller       3284
Action         2676
Romance        2541
Crime          1959
Horror         1888
Adventure      1807
Sci-Fi         1495
Fantasy        1205
Children       1081
Animation      1068
Mystery        1050
Documentary     946
War             670
Musical         532
Western         325
IMAX            173
Film-Noir       164
dtype: int64

### Weighting and normalising
Every movie shares "Drama" with half the catalogue, so sharing it says little. Sharing "Film-Noir" says a lot. Each column is multiplied by its IDF, `log(N / movies that have the feature)`, so common features count less and rare ones count more. Then each row is scaled to length 1, so that a dot product between two movies is their cosine similarity.

In [27]:
B   = pd.concat([genres, decades], axis=1)                  # binary features, 17,241 x 31
idf = np.log(n_items / B.sum(axis=0))                       # rarer column -> larger weight

X    = B.values * idf.values                                # weight each column
norm = np.linalg.norm(X, axis=1, keepdims=True)
X    = np.divide(X, norm, out=np.zeros_like(X), where=norm > 0)   # rows of all zeros stay zero

print("feature matrix:", X.shape, "| movies with no features at all:", int((norm == 0).sum()))
idf.sort_values().round(2)

feature matrix: (17241, 31) | movies with no features at all: 15


Drama          0.76
Comedy         1.09
2010s          1.34
2000s          1.41
Thriller       1.66
1990s          1.79
Action         1.86
Romance        1.91
Crime          2.17
Horror         2.21
Adventure      2.26
1980s          2.30
Sci-Fi         2.45
Fantasy        2.66
Children       2.77
Animation      2.78
Mystery        2.80
Documentary    2.90
1970s          2.91
2020s          3.03
1960s          3.10
War            3.25
Musical        3.48
1950s          3.48
1940s          3.78
Western        3.97
1930s          4.23
IMAX           4.60
Film-Noir      4.66
1920s          5.11
1910s          5.84
dtype: float64

### How distinguishable are movies?
With only genres and decade, many movies end up with the exact same vector. A model can't rank identical vectors against each other, so this limits what it can do. This cell counts how many movies share their vector with at least one other movie, by rarity bucket.

In [28]:
vec_id = pd.util.hash_pandas_object(B, index=False)         # one hash per row; same row -> same hash
same   = vec_id.map(vec_id.value_counts()) - 1              # how many OTHER movies share this vector

print("distinct vectors:", vec_id.nunique(), "of", n_items, "movies")
print("sizes of the 5 biggest groups of identical movies:", vec_id.value_counts().head(5).tolist())

meta["shared"] = same > 0
meta.groupby("bucket", observed=False).agg(
    movies=("title", "size"),
    pct_sharing_a_vector=("shared", lambda s: round(100 * s.mean(), 1)),
)

distinct vectors: 3052 of 17241 movies
sizes of the 5 biggest groups of identical movies: [587, 531, 464, 391, 329]


,movies,pct_sharing_a_vector
bucket,,
0,1095,92.1
1-4,5666,90.9
5-19,4836,91.5
20-99,3299,89.1
100-499,1685,86.8
500+,660,81.2


## 3 · User profiles and content scores
A user's profile is the average feature vector of the movies they liked in train (rating ≥ 4). A movie's score for that user is the dot product of the profile with the movie's vector, so movies that look like what the user liked score highest. We hide movies the user already rated and take the top 10, using the same ranking and metric code as notebooks 03 and 04. First cell: imports and the "already seen" matrix.

In [29]:
import scipy.sparse as sp
from recutils import rank_topk, evaluate_topk

def seen_matrix(*dfs):
    """Sparse 0/1 matrix of (user, movie) pairs a user already has; used to hide them from recommendations."""
    d = pd.concat(dfs)
    return sp.csr_matrix((np.ones(len(d), dtype=np.float32), (d.u.values, d.i.values)),
                         shape=(n_users, n_items))

seen_train = seen_matrix(train)
rel_val = to_sets(val_rel)          # user -> set of validation movies they rated >= 4
print("validation users with >=1 relevant movie:", len(rel_val))

validation users with >=1 relevant movie: 18392


### Building the profiles
`L` marks which movies each user liked in train. Dividing each row by the user's number of liked movies and multiplying by `X` gives the average vector of their liked movies. Users with no liked movie get a profile of all zeros, so every movie scores 0 for them.

In [30]:
def build_profiles(df):
    liked = df[df.rating >= REL]
    L = sp.csr_matrix((np.ones(len(liked), dtype=np.float32), (liked.u.values, liked.i.values)),
                      shape=(n_users, n_items))                      # users x movies, 1 = liked
    n_liked = np.asarray(L.sum(axis=1)).ravel()
    inv = np.divide(1.0, n_liked, out=np.zeros_like(n_liked), where=n_liked > 0)
    return sp.diags(inv) @ L @ X, n_liked                            # mean of liked-movie vectors

profiles, n_liked = build_profiles(train)
print("profiles:", profiles.shape, "| users with no liked movie in train:", int((n_liked == 0).sum()))

profiles: (20093, 31) | users with no liked movie in train: 34


### Scoring and evaluating on validation
Scores come from `profile · movie vector`. Many movies have identical vectors, so many scores tie exactly. We add a tiny random number (about 1e-6) to every score so ties are broken randomly rather than by movie index. The seeded generator makes the run reproducible. Popularity is re-run next to it as the reference.

In [31]:
rng = np.random.default_rng(SEED)
X32 = X.astype(np.float32)

def content_scores(ub, profiles):
    s = np.asarray(profiles[ub], dtype=np.float32) @ X32.T           # (batch, n_items)
    return s + 1e-6 * rng.random(s.shape, dtype=np.float32)          # random tie-break

ranked = rank_topk(lambda ub: content_scores(ub, profiles), rel_val.keys(), seen_train, K)
content_val = evaluate_topk(ranked, rel_val, n_items, K)

counts = np.bincount(train.i, minlength=n_items).astype(np.float32)
pop_val = evaluate_topk(rank_topk(lambda ub: np.tile(counts, (len(ub), 1)), rel_val.keys(), seen_train, K),
                        rel_val, n_items, K)

pd.DataFrame({"popularity": pop_val, "content (genres + decade)": content_val}).T.round(4)

,Recall@10,NDCG@10,HitRate@10,Coverage
popularity,0.0574,0.0415,0.1620,0.0137
content (genres + decade),0.0071,0.0047,0.0247,0.5449


### Looking at one user
A single user is an anecdote, but it shows what the model is doing. We print what the user liked, the top features of their profile, and the 10 movies recommended.

In [32]:
u = list(rel_val.keys())[0]
liked_u = train[(train.u == u) & (train.rating >= REL)].i
print("liked in train (first 8):", meta.loc[liked_u.head(8), "title"].tolist())

prof = profiles[u]
print("top profile features:", [(B.columns[j], round(float(prof[j]), 2)) for j in np.argsort(-prof)[:5]])
meta.loc[ranked[u], ["title", "genres"]]

liked in train (first 8): ['Being John Malkovich (1999)', 'Three Kings (1999)', 'Blade Runner (1982)', 'Emma (1996)', 'Trust (1990)', 'Shakespeare in Love (1998)', "Ferris Bueller's Day Off (1986)", 'Yojimbo (1961)']
top profile features: [('1990s', 0.25), ('1980s', 0.18), ('Drama', 0.16), ('Romance', 0.14), ('Comedy', 0.14)]


,title,genres
i,,
3648,Prelude to a Kiss (1992),Comedy|Drama|Romance
12575,"Same Love, Same Rain (El mismo amor, la misma ...",Comedy|Drama|Romance
7790,Venus Beauty Institute (Vénus beauté) (1999),Comedy|Drama|Romance
347,It Could Happen to You (1994),Comedy|Drama|Romance
1678,Hope Floats (1998),Comedy|Drama|Romance
1080,Delicatessen (1991),Comedy|Drama|Romance
2941,Singles (1992),Comedy|Drama|Romance
827,Twelfth Night (1996),Comedy|Drama|Romance
3473,"Couch in New York, A (1996)",Comedy|Drama|Romance


## 4 · Judging the content model on rare movies
Overall metrics barely reflect rare movies (they hold about 4% of relevant interactions), so we score the cold and tail slices separately, using validation only. Two views: (B) models rank only the slice's movies against each other, with random and popularity as baselines, and (A) the usual top 10 over all movies, counting only the rare relevant movies that show up. Each number comes with a 95% confidence interval from resampling users, because the slices are small. This first cell defines the helpers.

In [33]:
def per_user(ranked, rel):
    """Same metrics as evaluate_topk, but one value per user so we can bootstrap them."""
    disc = 1 / np.log2(np.arange(2, K + 2))
    rec, ndcg = [], []
    for u, r in rel.items():
        top = ranked[u][:K]
        g = np.array([m in r for m in top], dtype=float)
        rec.append(g.sum() / len(r))
        ndcg.append((g * disc[:len(top)]).sum() / disc[:min(len(r), K)].sum())
    return {"Recall@10": np.array(rec), "NDCG@10": np.array(ndcg)}

def slice_rel(rel, mask):
    """Keep only each user's relevant movies that fall in the slice; drop users left with none."""
    out = {u: {m for m in r if mask[m]} for u, r in rel.items()}
    return {u: r for u, r in out.items() if r}

def content_fn(ub):
    s = np.asarray(profiles[ub], dtype=np.float32) @ X32.T
    return s + 1e-6 * rng.random(s.shape, dtype=np.float32)

def pop_fn(ub):
    return np.tile(counts, (len(ub), 1)) + 1e-3 * rng.random((len(ub), n_items), dtype=np.float32)

def random_fn(ub):
    return rng.random((len(ub), n_items), dtype=np.float32)

MODELS = {"random": random_fn, "popularity": pop_fn, "content": content_fn}

### Running both views
`eval_slice` ranks for the users in a slice. With `restrict=True` every movie outside the slice gets a score of minus infinity, so only the slice's movies can be recommended (view B). With `restrict=False` all movies compete as normal (view A). `report` bootstraps over users and shows each mean with its 95% interval, plus the paired difference of content against the baselines.

Tie-breaking is random, so each model is run with 10 fixed seeds and the per-user scores are averaged before bootstrapping

In [34]:
N_TIE_SEEDS = 10

def eval_slice(slice_name, restrict, names):
    global rng                                   # the model functions read this generator
    mask = in_slice[("val", slice_name)]
    rel = slice_rel(rel_val, mask)
    out = {}
    for name in names:
        fn = MODELS[name]
        f = (lambda ub, fn=fn: np.where(mask, fn(ub), -np.inf)) if restrict else fn
        runs = []
        for seed in range(N_TIE_SEEDS):
            rng = np.random.default_rng(seed)    # fresh, fixed generator per run
            runs.append(per_user(rank_topk(f, rel.keys(), seen_train, K), rel))
        out[name] = {met: np.mean([r[met] for r in runs], axis=0) for met in runs[0]}
    return out, len(rel)

def ci(vals, idx):
    m = vals[idx].mean(axis=1)
    return vals.mean(), *np.percentile(m, [2.5, 97.5])

def report(restrict, names):
    brng = np.random.default_rng(SEED)
    for s in SLICES:
        res, n = eval_slice(s, restrict, names)
        idx = brng.integers(0, n, (1000, n))                 # same resampled users for every model
        rows = {name: {met: "%.4f [%.4f, %.4f]" % ci(d[met], idx) for met in d} for name, d in res.items()}
        print(f"{s} slice, {n} users")
        display(pd.DataFrame(rows).T)
        for base in names:
            if base != "content":
                diff = res["content"]["NDCG@10"] - res[base]["NDCG@10"]
                print("  NDCG content - %s: %.4f [%.4f, %.4f]" % ((base,) + ci(diff, idx)))

### View B: ranking only within the slice
Which model orders rare movies best? Random is the floor.

In [35]:
report(restrict=True, names=["random", "popularity", "content"])

cold slice, 676 users


,Recall@10,NDCG@10
random,"0.0014 [0.0006, 0.0023]","0.0007 [0.0003, 0.0011]"
popularity,"0.0044 [0.0000, 0.0104]","0.0036 [0.0000, 0.0087]"
content,"0.0136 [0.0063, 0.0220]","0.0057 [0.0027, 0.0093]"


  NDCG content - random: 0.0051 [0.0021, 0.0086]
  NDCG content - popularity: 0.0021 [-0.0039, 0.0074]
tail slice, 2144 users


,Recall@10,NDCG@10
random,"0.0010 [0.0006, 0.0014]","0.0005 [0.0003, 0.0007]"
popularity,"0.0041 [0.0019, 0.0069]","0.0018 [0.0008, 0.0031]"
content,"0.0104 [0.0072, 0.0139]","0.0053 [0.0036, 0.0072]"


  NDCG content - random: 0.0048 [0.0030, 0.0067]
  NDCG content - popularity: 0.0035 [0.0012, 0.0057]


### View A: top 10 over all movies, counting only rare relevant movies
Popularity can't recommend rare movies by design, so it should score 0 here. This shows whether content features surface rare movies at all when competing with popular ones.

In [36]:
report(restrict=False, names=["popularity", "content"])

cold slice, 676 users


,Recall@10,NDCG@10
popularity,"0.0000 [0.0000, 0.0000]","0.0000 [0.0000, 0.0000]"
content,"0.0046 [0.0004, 0.0099]","0.0016 [0.0002, 0.0035]"


  NDCG content - popularity: 0.0016 [0.0002, 0.0035]
tail slice, 2144 users


,Recall@10,NDCG@10
popularity,"0.0000 [0.0000, 0.0000]","0.0000 [0.0000, 0.0000]"
content,"0.0067 [0.0040, 0.0099]","0.0031 [0.0020, 0.0045]"


  NDCG content - popularity: 0.0031 [0.0020, 0.0045]


## Observations (steps 3 and 4)
- Genres and decade give 3,052 distinct vectors for 17,241 movies, and about 91% of rare movies share a vector with another movie. Ties are broken randomly, and every slice result averages 10 tie-break seeds.
- Over all movies, content alone is far below popularity (validation NDCG@10 0.005 vs 0.042) but recommends a much wider range of movies (coverage 0.54 vs 0.014).
- Within the tail slice (<20 positives, 2,144 users), content gets NDCG@10 0.0053 against 0.0018 for popularity and 0.0005 for random. Content minus popularity is +0.0035 [0.0012, 0.0057].
- Within the cold slice (<5 positives, 676 users), content gets 0.0057 against 0.0036 for popularity. The difference is +0.0021 [-0.0039, 0.0074], so this sample can't separate them. Both are above random (0.0007).
- When competing against all movies, popularity never recommends a rare movie. Content gets NDCG@10 0.0016 (cold) and 0.0031 (tail): small, but above zero.
- Recall@10 within the tail slice is about 1%, so only about 1 in 100 of a user's relevant rare movies reaches their top 10. With so many identical vectors that is expected, and it is why tags come next.
- A single tie-break seed moved the cold numbers noticeably between runs, so single-run results on small slices shouldn't be trusted.

**Next:** tags built from users outside the sample, then a blend with the ALS model.